# GCP Vertex AI TPM 할당량 소모 방식 검증 핸즈온

## 목적

이 노트북은 GCP Vertex AI 환경에서 **TPM(Tokens Per Minute) 할당량이 실제로 어떻게 차감되는지** 직접 검증하기 위한 테스트입니다.

특히, **캐시된 토큰(Cached Tokens)이 TPM 할당량에서 할인 없이 1:1로 차감되는지** 여부를 확인합니다.

### 검증 결과의 활용

이 테스트를 통해 확인한 `usageMetadata`의 정확한 토큰 카운트 구조는 할당량 제한 로직을 설계하는 핵심 기준이 됩니다.

### 테스트 흐름

| 단계 | 내용 | 방법 |
|------|------|------|
| **1단계** | 12k 캐시 생성 및 다중 호출 테스트 | Python 코드 실행 |
| **2단계** | GCP 콘솔에서 TPM 차감 그래프 확인 | 콘솔 모니터링 |

---
## 사전 준비

### 필수 패키지 설치

Vertex AI SDK가 설치되어 있지 않은 경우 아래 셀을 실행하세요.

In [ ]:
!pip install --upgrade google-cloud-aiplatform google-cloud-monitoring

### GCP 인증

| 실행 환경 | 인증 방법 |
|-----------|-----------|
| **Colab Enterprise** | 별도 인증 불필요 (실행자의 IAM 권한이 자동 적용) |
| **Vertex AI Workbench / Cloud Shell** | 별도 인증 불필요 |
| **Google Colab (무료)** | 아래 셀에서 `auth.authenticate_user()` 주석 해제 후 실행 |
| **로컬 환경** | 아래 셀에서 `gcloud auth` 주석 해제 후 실행 |

> **참고:** Colab Enterprise는 GCP 프로젝트 내에서 실행되므로 사용자의 IAM 권한이 자동으로 적용됩니다.  
> 단, 실행자에게 **Vertex AI User** 역할(또는 동등 권한)이 부여되어 있어야 합니다.

In [ ]:
# ──────────────────────────────────────────────────
# Colab Enterprise / Workbench / Cloud Shell → 이 셀 건너뛰기
# ──────────────────────────────────────────────────

# Google Colab (무료)에서 실행하는 경우:
# from google.colab import auth
# auth.authenticate_user()

# 로컬 환경에서 실행하는 경우:
# !gcloud auth application-default login

---
## 1단계: Python 코드로 12k 캐시 생성 및 다중 호출 테스트

게이트웨이를 거치지 않고 **Vertex AI SDK를 직접 호출**하여 순수한 베이스라인 지표를 발생시킵니다.

### 1-1. 환경 변수 설정

> **아래 `PROJECT_ID`를 본인 GCP 프로젝트 ID로 반드시 수정하세요.**

In [ ]:
PROJECT_ID = "YOUR_PROJECT_ID"  # ← 본인 프로젝트 ID로 변경
LOCATION = "us-central1"
MODEL_ID = "gemini-2.5-flash"  # Gemini 2.5 Flash

# 테스트 파라미터
NUM_CALLS = 10           # API 호출 횟수
CACHE_TTL_HOURS = 1      # 캐시 유지 시간
SYSTEM_PROMPT_REPEAT = 500  # 시스템 프롬프트 반복 횟수 (약 10k~12k 토큰 생성용)

print(f"프로젝트: {PROJECT_ID}")
print(f"리전: {LOCATION}")
print(f"모델: {MODEL_ID}")
print(f"호출 횟수: {NUM_CALLS}회")

### 1-2. Vertex AI 초기화

In [ ]:
import datetime
import time

import vertexai
from vertexai.generative_models import GenerativeModel
from vertexai.preview import caching

vertexai.init(project=PROJECT_ID, location=LOCATION)
print("Vertex AI 초기화 완료")

### 1-2a. Cloud Monitoring 메트릭 조회 설정

Google Cloud Monitoring API를 사용하여 Vertex AI의 토큰 관련 메트릭을 프로그래밍 방식으로 조회합니다.

| 메트릭 | 설명 |
|--------|------|
| `token_count` | 온라인 서빙에서 사용된 토큰 수 |
| `consumed_token_throughput` | 번다운 레이트를 반영한 실제 토큰 처리량 (TPM 계산 가능) |

In [ ]:
from google.cloud import monitoring_v3
from google.api_core import client_options as client_options_lib
from datetime import timedelta

METRIC_TOKEN_COUNT = "aiplatform.googleapis.com/publisher/online_serving/token_count"
METRIC_CONSUMED_THROUGHPUT = "aiplatform.googleapis.com/publisher/online_serving/consumed_token_throughput"

monitoring_client = monitoring_v3.MetricServiceClient(
    client_options=client_options_lib.ClientOptions(
        quota_project_id=PROJECT_ID,
    ),
)


def query_metric(metric_type, start_time, end_time, alignment_seconds=60,
                 aligner=monitoring_v3.Aggregation.Aligner.ALIGN_DELTA):
    """Cloud Monitoring에서 Vertex AI 메트릭을 조회합니다."""
    project_name = f"projects/{PROJECT_ID}"

    interval = monitoring_v3.TimeInterval(
        start_time=start_time,
        end_time=end_time,
    )

    aggregation = monitoring_v3.Aggregation(
        alignment_period={"seconds": alignment_seconds},
        per_series_aligner=aligner,
    )

    filter_str = (
        f'metric.type = "{metric_type}"'
        f' AND resource.labels.model_user_id = "{MODEL_ID}"'
        f' AND resource.labels.location = "{LOCATION}"'
    )

    try:
        return list(monitoring_client.list_time_series(
            request={
                "name": project_name,
                "filter": filter_str,
                "interval": interval,
                "aggregation": aggregation,
                "view": monitoring_v3.ListTimeSeriesRequest.TimeSeriesView.FULL,
            }
        ))
    except Exception as e:
        print(f"  조회 오류: {e}")
        return []


def get_point_value(point):
    """TypedValue에서 숫자 값을 추출합니다."""
    v = point.value
    return v.double_value if v.double_value != 0 else v.int64_value


def format_ts(t):
    """타임스탬프를 문자열로 변환합니다."""
    if hasattr(t, 'strftime'):
        return t.strftime('%H:%M:%S')
    elif hasattr(t, 'seconds'):
        dt_obj = datetime.datetime.fromtimestamp(t.seconds, tz=datetime.timezone.utc)
        return dt_obj.strftime('%H:%M:%S')
    return str(t)


def display_metric(time_series_list, label=""):
    """메트릭 타임시리즈 데이터를 출력합니다."""
    if not time_series_list:
        print(f"  [{label}] 데이터 없음")
        print(f"    -> 지표 반영까지 2~5분 지연될 수 있습니다. 잠시 후 다시 실행하세요.")
        return []

    all_values = []
    for ts in time_series_list:
        labels = dict(ts.metric.labels)
        print(f"\n  [{label}] 메트릭 라벨: {labels}")
        print(f"  데이터 포인트: {len(ts.points)}개")

        for pt in sorted(ts.points, key=lambda p: p.interval.end_time):
            val = get_point_value(pt)
            t = pt.interval.end_time
            print(f"    {format_ts(t)} UTC -> {val:>12,.2f}")
            all_values.append({"time": t, "value": val, "labels": labels})

    return all_values


def fetch_run_metrics(start_time, end_time, run_label):
    """시작/종료 시점 및 전체 기간의 token_count, consumed_token_throughput을 조회합니다."""
    print(f"\n{'━' * 70}")
    print(f"  Cloud Monitoring 메트릭 조회 - {run_label}")
    print(f"  테스트 시작: {start_time.strftime('%Y-%m-%d %H:%M:%S')} UTC")
    print(f"  테스트 종료: {end_time.strftime('%Y-%m-%d %H:%M:%S')} UTC")
    print(f"{'━' * 70}")

    metrics = [
        ("token_count", METRIC_TOKEN_COUNT),
        ("consumed_token_throughput", METRIC_CONSUMED_THROUGHPUT),
    ]

    results = {}

    for name, mtype in metrics:
        print(f"\n{'─' * 70}")
        print(f"  {name}")
        print(f"  ({mtype})")
        print(f"{'─' * 70}")

        # 시작 시점 (전후 2분)
        print(f"\n  [시작 시점] {start_time.strftime('%H:%M:%S')} UTC 전후 2분")
        s_data = query_metric(
            mtype,
            start_time - timedelta(minutes=2),
            start_time + timedelta(minutes=2),
        )
        s_vals = display_metric(s_data, f"{name} (시작)")

        # 종료 시점 (전후 2분)
        print(f"\n  [종료 시점] {end_time.strftime('%H:%M:%S')} UTC 전후 2분")
        e_data = query_metric(
            mtype,
            end_time - timedelta(minutes=2),
            end_time + timedelta(minutes=2),
        )
        e_vals = display_metric(e_data, f"{name} (종료)")

        # 전체 기간 (시작 1분 전 ~ 종료 5분 후)
        print(f"\n  [전체 기간] {start_time.strftime('%H:%M:%S')} ~ {end_time.strftime('%H:%M:%S')} UTC (+버퍼)")
        f_data = query_metric(
            mtype,
            start_time - timedelta(minutes=1),
            end_time + timedelta(minutes=5),
        )
        f_vals = display_metric(f_data, f"{name} (전체)")

        results[name] = {
            "start": s_vals,
            "end": e_vals,
            "full": f_vals,
            "raw": f_data,
        }

    return results


print("Cloud Monitoring 헬퍼 함수 정의 완료")
print(f"  필터: model_user_id={MODEL_ID}, location={LOCATION}")

In [ ]:
print("프로젝트에서 사용 가능한 Vertex AI publisher 메트릭 탐색 중...\n")

# 1. 메트릭 디스크립터 확인
for metric_type in [METRIC_TOKEN_COUNT, METRIC_CONSUMED_THROUGHPUT]:
    try:
        descriptor = monitoring_client.get_metric_descriptor(
            name=f"projects/{PROJECT_ID}/metricDescriptors/{metric_type}"
        )
        print(f"[OK] {descriptor.display_name}")
        print(f"  Type: {descriptor.type}")
        print(f"  Kind: {descriptor.metric_kind.name}, Value: {descriptor.value_type.name}")
        print(f"  Metric Labels: {[l.key for l in descriptor.labels]}")
    except Exception as e:
        print(f"[FAIL] {metric_type}")
        print(f"  {e}\n")

# 2. 최근 1시간 데이터 조회 (모델 필터 없이 → 실제 라벨 값 확인)
print(f"\n{'─' * 70}")
print("최근 1시간 데이터 조회 (모델 필터 없이, HEADERS만)...")
print(f"{'─' * 70}")

end_time = datetime.datetime.now(datetime.timezone.utc)
start_time = end_time - timedelta(hours=1)

for metric_name, metric_type in [("token_count", METRIC_TOKEN_COUNT),
                                  ("consumed_throughput", METRIC_CONSUMED_THROUGHPUT)]:
    try:
        results = list(monitoring_client.list_time_series(
            request={
                "name": f"projects/{PROJECT_ID}",
                "filter": f'metric.type = "{metric_type}"',
                "interval": monitoring_v3.TimeInterval(
                    start_time=start_time,
                    end_time=end_time,
                ),
                "view": monitoring_v3.ListTimeSeriesRequest.TimeSeriesView.HEADERS,
            }
        ))
        if results:
            print(f"\n[{metric_name}] {len(results)}개 시리즈 발견")
            for ts in results[:5]:
                print(f"  Resource Type : {ts.resource.type}")
                print(f"  Resource Labels: {dict(ts.resource.labels)}")
                print(f"  Metric Labels : {dict(ts.metric.labels)}")
                print()
        else:
            print(f"\n[{metric_name}] 데이터 없음 (최근 1시간 내 해당 메트릭 없음)")
    except Exception as e:
        print(f"\n[{metric_name}] 조회 실패: {e}")

### 1-3. 대규모 시스템 프롬프트(캐시) 생성

약 **10,000 ~ 12,000 토큰** 분량의 더미 텍스트를 시스템 프롬프트로 사용하여 캐시를 생성합니다.  
이 캐시가 이후 모든 호출에서 재활용되며, TPM 할당량에 어떻게 반영되는지가 핵심 검증 포인트입니다.

In [ ]:
print("대규모 시스템 프롬프트(캐시) 생성 중...")

long_system_instruction = (
    "당신은 사내 보안 규정과 아키텍처를 안내하는 전문 AI 어시스턴트입니다. "
    * SYSTEM_PROMPT_REPEAT
)

print(f"시스템 프롬프트 길이: {len(long_system_instruction):,} 글자")
print(f"예상 토큰 수: 약 {len(long_system_instruction) // 4:,} 토큰 (추정)")

In [ ]:
cached_content = caching.CachedContent.create(
    model_name=MODEL_ID,
    system_instruction=long_system_instruction,
    ttl=datetime.timedelta(hours=CACHE_TTL_HOURS),
)

print(f"캐시 생성 완료!")
print(f"  캐시 이름: {cached_content.name}")
print(f"  캐시 만료: {CACHE_TTL_HOURS}시간 후")

### 1-4. 캐시를 활용한 10회 연속 API 호출

사내 개발자가 던질 법한 짧은 질문을 시뮬레이션하여 10회 연속 호출합니다.  
각 호출마다 `usageMetadata`에서 토큰 수치를 추출합니다.

#### 확인 포인트
- `prompt_token_count` (Total Input): 매 호출마다 **약 12,000대 이상**으로 나오는지 확인
- `cached_content_token_count`: 캐시된 토큰이 별도로 표기되는지 확인
- 이 `prompt_token_count` 전체가 Vertex AI가 인식하는 **1회 호출당 Input 토큰량**입니다

In [ ]:
print(f"캐시를 활용한 {NUM_CALLS}회 연속 API 호출 시작...\n")
print("=" * 65)

model = GenerativeModel.from_cached_content(cached_content=cached_content)

results = []

cached_start_time = datetime.datetime.now(datetime.timezone.utc)
print(f"  >> 시작 시각 (UTC): {cached_start_time.isoformat()}\n")

for i in range(1, NUM_CALLS + 1):
    response = model.generate_content("클라우드 보안 설정에 대해 요약해 줘.")

    usage = response.usage_metadata

    result = {
        "call_number": i,
        "prompt_token_count": usage.prompt_token_count,
        "cached_content_token_count": usage.cached_content_token_count,
        "candidates_token_count": usage.candidates_token_count,
    }
    results.append(result)

    print(f"--- [호출 {i:2d}/{NUM_CALLS}] ---")
    print(f"  Total Input  (prompt_token_count)          : {usage.prompt_token_count:>8,}")
    print(f"  └ Cached     (cached_content_token_count)  : {usage.cached_content_token_count:>8,}")
    print(f"  Output       (candidates_token_count)      : {usage.candidates_token_count:>8,}")
    print()

cached_end_time = datetime.datetime.now(datetime.timezone.utc)
print(f"  >> 종료 시각 (UTC): {cached_end_time.isoformat()}")
print("=" * 65)
print(f"{NUM_CALLS}회 호출 완료!")

### 1-5. 결과 요약 및 분석

In [ ]:
total_prompt_tokens = sum(r["prompt_token_count"] for r in results)
total_cached_tokens = sum(r["cached_content_token_count"] for r in results)
total_output_tokens = sum(r["candidates_token_count"] for r in results)
avg_prompt_per_call = total_prompt_tokens / len(results)

print("━" * 65)
print("                    결과 요약")
print("━" * 65)
print(f"  총 호출 횟수                          : {len(results):>10}회")
print(f"  1회 평균 Input 토큰 (prompt_token)    : {avg_prompt_per_call:>10,.0f}")
print(f"  총 Input 토큰 합계                    : {total_prompt_tokens:>10,}")
print(f"  총 Cached 토큰 합계                   : {total_cached_tokens:>10,}")
print(f"  총 Output 토큰 합계                   : {total_output_tokens:>10,}")
print("━" * 65)
print()
print("▶ 예상 TPM 차감량 (Input만):")
print(f"  prompt_token_count × {NUM_CALLS}회 = {total_prompt_tokens:,} 토큰")
print()
print("▶ 만약 캐시가 할인된다면 (Non-cached만 차감 시):")
non_cached_total = total_prompt_tokens - total_cached_tokens
print(f"  (prompt - cached) × {NUM_CALLS}회 = {non_cached_total:,} 토큰")
print()
print("━" * 65)
print("↑ GCP 콘솔 할당량 그래프에서 실제 TPM 피크가")
print(f"  {total_prompt_tokens:,}에 가까우면 → 캐시 토큰도 1:1로 TPM 차감")
print(f"  {non_cached_total:,}에 가까우면 → 캐시 토큰은 TPM에서 할인")
print("━" * 65)

### 1-6a. Cloud Monitoring 메트릭 조회 (캐시 사용)

캐시를 사용한 테스트 구간의 `token_count`와 `consumed_token_throughput` 메트릭을 Cloud Monitoring API로 직접 조회합니다.

각 메트릭에 대해 **시작 시점**, **종료 시점**, **전체 기간** 3가지 구간으로 나누어 조회합니다.

> **참고:** Cloud Monitoring 지표는 반영까지 **2~5분** 지연될 수 있습니다.
> 데이터가 없으면 잠시 기다린 후 이 셀을 다시 실행하세요.

In [ ]:
cached_metrics = fetch_run_metrics(cached_start_time, cached_end_time, "캐시 사용")

### 1-6. 호출별 토큰 사용량 시각화

In [ ]:
try:
    import matplotlib.pyplot as plt
    import matplotlib.ticker as ticker

    calls = [r["call_number"] for r in results]
    prompts = [r["prompt_token_count"] for r in results]
    cached = [r["cached_content_token_count"] for r in results]
    outputs = [r["candidates_token_count"] for r in results]

    fig, ax = plt.subplots(figsize=(12, 5))

    bar_width = 0.25
    x = range(len(calls))
    x1 = [i - bar_width for i in x]
    x2 = list(x)
    x3 = [i + bar_width for i in x]

    ax.bar(x1, prompts, width=bar_width, label="Total Input (prompt_token_count)", color="#4285F4")
    ax.bar(x2, cached, width=bar_width, label="Cached (cached_content_token_count)", color="#FBBC04")
    ax.bar(x3, outputs, width=bar_width, label="Output (candidates_token_count)", color="#34A853")

    ax.set_xlabel("호출 번호")
    ax.set_ylabel("토큰 수")
    ax.set_title("호출별 토큰 사용량 비교")
    ax.set_xticks(list(x))
    ax.set_xticklabels([f"#{c}" for c in calls])
    ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda v, _: f"{v:,.0f}"))
    ax.legend()
    ax.grid(axis="y", alpha=0.3)

    plt.tight_layout()
    plt.show()

except ImportError:
    print("matplotlib가 설치되어 있지 않습니다.")
    print("시각화를 원하시면: !pip install matplotlib")

### 1-7. (비교군) 캐시 없이 동일한 시스템 프롬프트로 10회 호출

캐시를 사용하지 않고 **매 호출마다 12k 시스템 프롬프트를 직접 전송**합니다.
GCP 콘솔에서 캐시 사용 시와 비캐시 시의 TPM 차감량을 비교하기 위한 베이스라인입니다.

> **참고:** 캐시 테스트와 시간대가 겹치지 않도록, 이 셀을 실행하기 전에 **약 2~3분 간격**을 두는 것을 권장합니다.

In [ ]:
print(f"(비교군) 캐시 없이 {NUM_CALLS}회 연속 API 호출 시작...\n")
print("=" * 65)

model_no_cache = GenerativeModel(
    model_name=MODEL_ID,
    system_instruction=long_system_instruction,
)

results_no_cache = []

no_cache_start_time = datetime.datetime.now(datetime.timezone.utc)
print(f"  >> 시작 시각 (UTC): {no_cache_start_time.isoformat()}\n")

for i in range(1, NUM_CALLS + 1):
    response = model_no_cache.generate_content("클라우드 보안 설정에 대해 요약해 줘.")

    usage = response.usage_metadata

    result = {
        "call_number": i,
        "prompt_token_count": usage.prompt_token_count,
        "cached_content_token_count": getattr(usage, "cached_content_token_count", 0) or 0,
        "candidates_token_count": usage.candidates_token_count,
    }
    results_no_cache.append(result)

    print(f"--- [호출 {i:2d}/{NUM_CALLS}] ---")
    print(f"  Total Input  (prompt_token_count)          : {usage.prompt_token_count:>8,}")
    print(f"  Cached       (cached_content_token_count)  : {result['cached_content_token_count']:>8,}")
    print(f"  Output       (candidates_token_count)      : {usage.candidates_token_count:>8,}")
    print()

no_cache_end_time = datetime.datetime.now(datetime.timezone.utc)
print(f"  >> 종료 시각 (UTC): {no_cache_end_time.isoformat()}")
print("=" * 65)
print(f"비캐시 {NUM_CALLS}회 호출 완료!")

### 1-8. 캐시 vs 비캐시 결과 비교

In [ ]:
nc_total_prompt = sum(r["prompt_token_count"] for r in results_no_cache)
nc_total_cached = sum(r["cached_content_token_count"] for r in results_no_cache)
nc_total_output = sum(r["candidates_token_count"] for r in results_no_cache)
nc_avg_prompt = nc_total_prompt / len(results_no_cache)

print("━" * 65)
print("         캐시 vs 비캐시 비교 요약")
print("━" * 65)
print()
print(f"  {'항목':<35} {'캐시 사용':>12} {'비캐시':>12}")
print(f"  {'─' * 35} {'─' * 12} {'─' * 12}")
print(f"  {'1회 평균 prompt_token_count':<35} {avg_prompt_per_call:>12,.0f} {nc_avg_prompt:>12,.0f}")
print(f"  {'총 Input 토큰 합계':<35} {total_prompt_tokens:>12,} {nc_total_prompt:>12,}")
print(f"  {'총 Cached 토큰 합계':<35} {total_cached_tokens:>12,} {nc_total_cached:>12,}")
print(f"  {'총 Output 토큰 합계':<35} {total_output_tokens:>12,} {nc_total_output:>12,}")
print()
print("━" * 65)
print("▶ GCP 콘솔에서 두 테스트의 TPM 스파이크를 비교하세요.")
print("  두 스파이크가 비슷하면 → 캐시 여부와 무관하게 prompt_token_count 전체가 TPM 차감")
print("━" * 65)

### 1-8a. Cloud Monitoring 메트릭 조회 (캐시 미사용)

캐시를 사용하지 않은 테스트 구간의 `token_count`와 `consumed_token_throughput` 메트릭을 조회합니다.

> **참고:** 캐시 사용 테스트와 메트릭 수치를 비교하여 캐시가 TPM 차감에 미치는 영향을 확인합니다.

In [ ]:
no_cache_metrics = fetch_run_metrics(no_cache_start_time, no_cache_end_time, "캐시 미사용")

### consumed_token_throughput에서 TPM 계산

`consumed_token_throughput`은 **번다운 레이트(burndown rate)를 반영한 실제 토큰 처리량**입니다.

#### TPM 계산 방법

| 방법 | Aligner | 결과 |
|------|---------|------|
| 방법 1 | `ALIGN_DELTA` + 60초 정렬 | 1분간 처리된 토큰 총량 = **TPM** |
| 방법 2 | `ALIGN_RATE` + 60초 정렬 | tokens/second x 60 = **TPM 추정치** |

> **결론:** `consumed_token_throughput`은 TPM 계산에 **직접 활용 가능**합니다.
> `ALIGN_DELTA`로 60초 정렬하면 각 데이터 포인트가 곧 해당 분의 TPM 값입니다.

In [ ]:
print("━" * 70)
print("  consumed_token_throughput -> TPM 계산")
print("━" * 70)


def calculate_tpm(start_time, end_time, label):
    """consumed_token_throughput에서 TPM을 계산합니다."""
    print(f"\n▶ {label}")
    query_start = start_time - timedelta(minutes=1)
    query_end = end_time + timedelta(minutes=5)

    # 방법 1: ALIGN_DELTA (60초) -> 1분간 총 토큰 = TPM
    print(f"\n  [방법 1] ALIGN_DELTA, 60초 정렬 (각 포인트 = 해당 분의 TPM)")
    delta_data = query_metric(
        METRIC_CONSUMED_THROUGHPUT, query_start, query_end,
        alignment_seconds=60,
        aligner=monitoring_v3.Aggregation.Aligner.ALIGN_DELTA,
    )

    tpm_values_delta = []
    if delta_data:
        for ts in delta_data:
            labels = dict(ts.metric.labels)
            print(f"    메트릭 라벨: {labels}")
            for pt in sorted(ts.points, key=lambda p: p.interval.end_time):
                val = get_point_value(pt)
                t = pt.interval.end_time
                tpm_values_delta.append(val)
                print(f"      {format_ts(t)} UTC -> TPM: {val:>12,.0f}")

        if tpm_values_delta:
            print(f"\n    Peak TPM: {max(tpm_values_delta):>12,.0f}")
            print(f"    Avg  TPM: {sum(tpm_values_delta)/len(tpm_values_delta):>12,.0f}")
    else:
        print("    데이터 없음 - 잠시 후 다시 실행하세요.")

    # 방법 2: ALIGN_RATE -> tokens/sec x 60 = TPM 추정
    print(f"\n  [방법 2] ALIGN_RATE (tokens/sec x 60 = TPM 추정)")
    rate_data = query_metric(
        METRIC_CONSUMED_THROUGHPUT, query_start, query_end,
        alignment_seconds=60,
        aligner=monitoring_v3.Aggregation.Aligner.ALIGN_RATE,
    )

    if rate_data:
        for ts in rate_data:
            labels = dict(ts.metric.labels)
            print(f"    메트릭 라벨: {labels}")
            for pt in sorted(ts.points, key=lambda p: p.interval.end_time):
                val = get_point_value(pt)
                t = pt.interval.end_time
                print(f"      {format_ts(t)} UTC -> {val:>10,.2f} tok/s -> TPM: {val * 60:>12,.0f}")
    else:
        print("    데이터 없음")

    return tpm_values_delta


# 캐시 사용 TPM
print()
try:
    cached_tpm = calculate_tpm(cached_start_time, cached_end_time, "캐시 사용 - TPM")
except NameError:
    print("  캐시 테스트를 먼저 실행하세요 (1-4 셀).")
    cached_tpm = []

# 캐시 미사용 TPM
print()
try:
    no_cache_tpm = calculate_tpm(no_cache_start_time, no_cache_end_time, "캐시 미사용 - TPM")
except NameError:
    print("  비캐시 테스트를 먼저 실행하세요 (1-7 셀).")
    no_cache_tpm = []

# 비교
print(f"\n{'━' * 70}")
print("  캐시 vs 비캐시 TPM 비교 (consumed_token_throughput 기반)")
print(f"{'━' * 70}")

if cached_tpm and no_cache_tpm:
    print(f"  캐시 사용   Peak TPM : {max(cached_tpm):>12,.0f}")
    print(f"  캐시 미사용 Peak TPM : {max(no_cache_tpm):>12,.0f}")
    print(f"\n  두 값이 비슷하면 -> 캐시 여부와 무관하게 TPM 1:1 차감")
    print(f"  캐시 사용 시 낮으면 -> 캐시 토큰은 TPM에서 할인")
else:
    print("  두 테스트를 모두 실행한 후 이 셀을 다시 실행하세요.")

print(f"{'━' * 70}")

---
## 2단계: GCP 콘솔에서 실시간 할당량(TPM) 차감 그래프 확인

코드를 실행하여 트래픽 스파이크를 발생시켰으니, 이제 **할당량 엔진이 이를 어떻게 계산했는지** 눈으로 직접 확인할 차례입니다.  
(데이터 액세스 로그를 켤 필요 없이 기본 시스템 지표로 확인 가능합니다.)

### 콘솔 확인 절차

#### Step 1. GCP 콘솔 이동
- **IAM 및 관리자 > 할당량 및 시스템 한도** 페이지로 접속합니다.

#### Step 2. 필터 설정
상단 필터(Filter) 창에 다음 두 가지 조건을 넣습니다:

| 필터 항목 | 값 |
|-----------|----|
| **서비스** | `Vertex AI API` |
| **할당량** | `generate_content_input_tokens_per_minute_per_base_model` |

> **참고:** `online_prediction_*` 메트릭은 3P 모델(Claude, Llama 등) 전용입니다.  
> 1P Gemini 모델은 `generateContent` API를 사용하므로 `generate_content_*` 메트릭에서 확인해야 합니다.

#### Step 3. 그래프 확인
- 필터링된 목록에서 방금 테스트한 **리전(`us-central1`)**과 **모델명(`gemini-2.5-flash`)**에 해당하는 항목을 찾습니다.
- 우측의 **[지표 보기(View Metrics)]** 아이콘을 클릭하여 차트 창을 엽니다.

#### Step 4. 결과 분석 (중요)
- 그래프의 시간 범위를 **'최근 1시간'**으로 좁힙니다.  
  (참고: 지표가 그래프에 반영되기까지 **약 2~3분 정도의 지연**이 있을 수 있습니다.)
- 그래프에서 **스파이크가 튄 정점(Peak) 수치**를 확인합니다.

### 판정 기준

아래 셀에서 계산된 값과 GCP 콘솔 그래프의 피크 값을 비교하세요.

In [ ]:
print("━" * 65)
print("           2단계: GCP 콘솔 그래프 비교 기준값")
print("━" * 65)
print()
print("GCP 콘솔 할당량 그래프의 피크 값과 아래 수치를 비교하세요:")
print()
print(f"  [A] 캐시 포함 전체 Input 토큰 합계 : {total_prompt_tokens:>10,}")
print(f"  [B] 캐시 제외 Input 토큰 합계      : {non_cached_total:>10,}")
print()
print("─" * 65)
print("  판정:")
print(f"  • 그래프 피크 ≈ [A] {total_prompt_tokens:,}")
print(f"    → 캐시 할인 없음! TPM은 prompt_token_count 기준 1:1 차감")
print()
print(f"  • 그래프 피크 ≈ [B] {non_cached_total:,}")
print(f"    → 캐시 토큰은 TPM에서 할인됨")
print("─" * 65)
print()
print("⏳ 참고: 지표 반영까지 2~3분 지연이 있을 수 있습니다.")
print("   콘솔에서 시간 범위를 '최근 1시간'으로 설정하세요.")

---
## 정리: 캐시 삭제

테스트가 끝나면 불필요한 캐시를 삭제하여 비용을 절약합니다.  
TTL을 1시간으로 설정했으므로 자동 만료되지만, 즉시 삭제하려면 아래 셀을 실행하세요.

In [ ]:
try:
    cached_content.delete()
    print(f"캐시 삭제 완료: {cached_content.name}")
except Exception as e:
    print(f"캐시 삭제 중 오류 (이미 만료되었을 수 있음): {e}")

---
## 결론 기록

아래 셀에 테스트 결과를 기록하세요.

In [ ]:
# ────────────────────────────────────────────
#  테스트 결과 기록 (직접 작성)
# ────────────────────────────────────────────

test_result = {
    "테스트 일시": "YYYY-MM-DD HH:MM",
    "테스트 모델": MODEL_ID,
    "리전": LOCATION,
    "호출 횟수": NUM_CALLS,
    "1회 평균 prompt_token_count": "여기에 기록",
    "GCP 콘솔 피크 값": "여기에 기록",
    "판정": "캐시 할인 없음(1:1 차감) / 캐시 할인 적용",  # 택 1
    "비고": "",
}

print("테스트 결과:")
for k, v in test_result.items():
    print(f"  {k}: {v}")

---
## 부록: 게이트웨이 프록시 설계 시 참고사항

위 테스트 결과를 기반으로 게이트웨이 프록시(LiteLLM 등)의 할당량 제한 로직을 설계할 때:

### 캐시가 1:1로 TPM에 차감되는 경우 (예상 시나리오)

```
TPM 소모량 = prompt_token_count (캐시 포함 전체) + candidates_token_count
```

- 프록시에서 Rate Limiting 계산 시 `cached_content_token_count`를 빼면 **안 됩니다**.
- BigQuery 로그 적재 시 `prompt_token_count` 전체를 TPM 차감 기준으로 기록해야 합니다.

### 캐시가 할인되는 경우

```
TPM 소모량 = (prompt_token_count - cached_content_token_count) + candidates_token_count
```

- 프록시에서 캐시 토큰을 차감한 값으로 Rate Limiting을 계산할 수 있습니다.